In [1]:
#import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#model development
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

#model evaluation
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [2]:
#full weather dataset
df = pd.read_csv("weather_prediction_dataset.csv")
# Preview the first few rows
df.head()

,DATE,MONTH,BASEL_cloud_cover,BASEL_humidity,BASEL_pressure,BASEL_global_radiation,BASEL_precipitation,BASEL_sunshine,BASEL_temp_mean,BASEL_temp_min,...,STOCKHOLM_temp_min,STOCKHOLM_temp_max,TOURS_wind_speed,TOURS_humidity,TOURS_pressure,TOURS_global_radiation,TOURS_precipitation,TOURS_temp_mean,TOURS_temp_min,TOURS_temp_max
0,20000101,1,8,0.89,1.0286,0.20,0.03,0.0,2.9,1.6,...,-9.3,0.7,1.6,0.97,1.0275,0.25,0.04,8.5,7.2,9.8
1,20000102,1,8,0.87,1.0318,0.25,0.00,0.0,3.6,2.7,...,0.5,2.0,2.0,0.99,1.0293,0.17,0.16,7.9,6.6,9.2
2,20000103,1,5,0.81,1.0314,0.50,0.00,3.7,2.2,0.1,...,-1.0,2.8,3.4,0.91,1.0267,0.27,0.00,8.1,6.6,9.6
3,20000104,1,7,0.79,1.0262,0.63,0.35,6.9,3.9,0.5,...,2.5,4.6,4.9,0.95,1.0222,0.11,0.44,8.6,6.4,10.8
4,20000105,1,5,0.90,1.0246,0.51,0.07,3.7,6.0,3.8,...,-1.8,2.9,3.6,0.95,1.0209,0.39,0.04,8.0,6.4,9.5


In [3]:
df.info()
df.describe()

#3,654 rows, 165 columns
#DATE is encoded as YYYYMMDD.

list(df.columns)
#=dataset has inconsistent variable availability across cities


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3654 entries, 0 to 3653
Columns: 165 entries, DATE to TOURS_temp_max
dtypes: float64(150), int64(15)
memory usage: 4.6 MB


['DATE',
 'MONTH',
 'BASEL_cloud_cover',
 'BASEL_humidity',
 'BASEL_pressure',
 'BASEL_global_radiation',
 'BASEL_precipitation',
 'BASEL_sunshine',
 'BASEL_temp_mean',
 'BASEL_temp_min',
 'BASEL_temp_max',
 'BUDAPEST_cloud_cover',
 'BUDAPEST_humidity',
 'BUDAPEST_pressure',
 'BUDAPEST_global_radiation',
 'BUDAPEST_precipitation',
 'BUDAPEST_sunshine',
 'BUDAPEST_temp_mean',
 'BUDAPEST_temp_max',
 'DE_BILT_cloud_cover',
 'DE_BILT_wind_speed',
 'DE_BILT_wind_gust',
 'DE_BILT_humidity',
 'DE_BILT_pressure',
 'DE_BILT_global_radiation',
 'DE_BILT_precipitation',
 'DE_BILT_sunshine',
 'DE_BILT_temp_mean',
 'DE_BILT_temp_min',
 'DE_BILT_temp_max',
 'DRESDEN_cloud_cover',
 'DRESDEN_wind_speed',
 'DRESDEN_wind_gust',
 'DRESDEN_humidity',
 'DRESDEN_global_radiation',
 'DRESDEN_precipitation',
 'DRESDEN_sunshine',
 'DRESDEN_temp_mean',
 'DRESDEN_temp_min',
 'DRESDEN_temp_max',
 'DUSSELDORF_cloud_cover',
 'DUSSELDORF_wind_speed',
 'DUSSELDORF_wind_gust',
 'DUSSELDORF_humidity',
 'DUSSELDORF_pres

In [4]:
# load data
df["DATE"] = pd.to_datetime(df["DATE"], format="%Y%m%d")

# calendar features
df["MONTH"] = df["DATE"].dt.month
df["DAYOFYEAR"] = df["DATE"].dt.dayofyear
df["SEASON"] = (df["MONTH"] % 12 // 3) + 1

# target columns and locations
target_cols = [col for col in df.columns if col.endswith("_temp_mean")]
locations = [col.replace("_temp_mean", "") for col in target_cols]

def find(location, variable):
    col = f"{location}_{variable}"
    return col if col in df.columns else None

In [5]:
#feature engineering
all_features = []   # store feature DataFrames

for location in locations:
    temp_mean = find(location, "temp_mean")
    temp_min = find(location, "temp_min")
    temp_max = find(location, "temp_max")
    humidity = find(location, "humidity")
    pressure = find(location, "pressure")
    wind_speed = find(location, "wind_speed")
    wind_gust = find(location, "wind_gust")

    feats = {}   # store features for this location

#lag features 
    if temp_mean:
        feats[f"{location}_temp_lag1"] = df[temp_mean].shift(1)
        feats[f"{location}_temp_lag2"] = df[temp_mean].shift(2)
        feats[f"{location}_temp_lag3"] = df[temp_mean].shift(3)

    if humidity:
        feats[f"{location}_humidity_lag1"] = df[humidity].shift(1)

    if pressure:
        feats[f"{location}_pressure_lag1"] = df[pressure].shift(1)

    if wind_speed:
        feats[f"{location}_wind_lag1"] = df[wind_speed].shift(1)

#rolling features 
    if temp_mean:
        feats[f"{location}_temp_roll3"] = (df[temp_mean].shift(1).rolling(3).mean())

    if humidity:
        feats[f"{location}_humidity_roll3"] = (df[humidity].shift(1).rolling(3).mean())

    if pressure:
        feats[f"{location}_pressure_roll5"] = (df[pressure].shift(1).rolling(5).mean())
        
# interaction features
    if temp_mean and humidity:
        feats[f"{location}_temp_humidity"] = (df[temp_mean].shift(1) * df[humidity].shift(1))

    if wind_speed and wind_gust:
        feats[f"{location}_wind_stress"] = (df[wind_speed].shift(1) * df[wind_gust].shift(1))

#difference features 
    if pressure:
        feats[f"{location}_pressure_change"] = (
            df[pressure].shift(1) - df[pressure].shift(2)
        )

    if temp_max and temp_min:
        feats[f"{location}_temp_diff"] = (
            df[temp_max].shift(1) - df[temp_min].shift(1)
        )

    if wind_speed and wind_gust:
        feats[f"{location}_wind_diff"] = (
            df[wind_gust].shift(1) - df[wind_speed].shift(1)
        )

    all_features.append(pd.DataFrame(feats, index=df.index))

# Combine all engineered features at once
engineered = pd.concat(all_features, axis=1)

print("Number of engineered features:", engineered.shape[1])
engineered.head()

Number of engineered features: 208


,BASEL_temp_lag1,BASEL_temp_lag2,BASEL_temp_lag3,BASEL_humidity_lag1,BASEL_pressure_lag1,BASEL_temp_roll3,BASEL_humidity_roll3,BASEL_pressure_roll5,BASEL_temp_humidity,BASEL_pressure_change,...,TOURS_temp_lag3,TOURS_humidity_lag1,TOURS_pressure_lag1,TOURS_wind_lag1,TOURS_temp_roll3,TOURS_humidity_roll3,TOURS_pressure_roll5,TOURS_temp_humidity,TOURS_pressure_change,TOURS_temp_diff
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.9,NaN,NaN,0.89,1.0286,NaN,NaN,NaN,2.581,NaN,...,NaN,0.97,1.0275,1.6,NaN,NaN,NaN,8.245,NaN,2.6
2,3.6,2.9,NaN,0.87,1.0318,NaN,NaN,NaN,3.132,0.0032,...,NaN,0.99,1.0293,2.0,NaN,NaN,NaN,7.821,0.0018,2.6
3,2.2,3.6,2.9,0.81,1.0314,2.900000,0.856667,NaN,1.782,-0.0004,...,8.5,0.91,1.0267,3.4,8.166667,0.956667,NaN,7.371,-0.0026,3.0
4,3.9,2.2,3.6,0.79,1.0262,3.233333,0.823333,NaN,3.081,-0.0052,...,7.9,0.95,1.0222,4.9,8.200000,0.950000,NaN,8.170,-0.0045,4.4


In [6]:
#feature matrix
X = pd.concat(
    [engineered, df[["MONTH", "DAYOFYEAR", "SEASON"]]],
    axis=1)

#multi-output target matrix
y = df[target_cols].copy()
dates = df["DATE"].copy()

#replace infinite values
X = X.replace([np.inf, -np.inf], np.nan)

#remove first 5 rows affected by lag and rolling features
X = X.iloc[5:].copy()
y = y.iloc[5:].copy()
dates = dates.iloc[5:].copy()

#keep rows where all target temperatures are available
valid = y.notna().all(axis=1)

X = X.loc[valid]
y = y.loc[valid]
dates = dates.loc[valid]

In [7]:
#chronological 80/20 split
X_train, X_test, y_train, y_test, dates_train, dates_test = train_test_split(
    X, y, dates, test_size=0.2, shuffle=False)

#remove features missing in training data
usable_cols = X_train.columns[X_train.notna().any()]
X_train = X_train[usable_cols]
X_test = X_test[usable_cols]

print("X train shape:", X_train.shape)
print("X test shape:", X_test.shape)
print("y train shape:", y_train.shape)
print("y test shape:", y_test.shape)

print("Training period:", dates_train.min(), "to", dates_train.max())
print("Testing period:", dates_test.min(), "to", dates_test.max())


X train shape: (2919, 211)
X test shape: (730, 211)
y train shape: (2919, 18)
y test shape: (730, 18)
Training period: 2000-01-06 00:00:00 to 2008-01-02 00:00:00
Testing period: 2008-01-03 00:00:00 to 2010-01-01 00:00:00


In [8]:
#Decision Tree
tree_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeRegressor(
        random_state=42
    ))
])

#Random Forest
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ))
])

tree_params = {
    "model__max_depth": [5, 10, 12, 15, 18, 20],
    "model__min_samples_split": [2, 4, 8, 12, 16],
    "model__min_samples_leaf": [1, 2, 4]
}

rf_params = {
    "model__n_estimators": [100, 200, 300, 400, 500],
    "model__max_depth": [10, 20, 30],
    "model__min_samples_split": [2, 4, 6, 8],
    "model__min_samples_leaf": [1, 2, 4]
}

cv = TimeSeriesSplit(n_splits=3)

tree_search = RandomizedSearchCV(
    estimator=tree_model,
    param_distributions=tree_params,
    n_iter=10,
    cv=cv,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_params,
    n_iter=10,
    cv=cv,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

models = {
    "Decision Tree": tree_search,
    "Random Forest": rf_search
}


In [ ]:
#Time-series cross-validation
cv = TimeSeriesSplit(n_splits=3)
cv_results = []

for name, model in models.items():
    
    scores = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=1)

    cv_results.append({
        "Model": name,
        "CV MSE": scores.mean()})

cv_results = pd.DataFrame(cv_results)

print(cv_results.round(2).to_string(index=False))


In [ ]:
#train & evaluate each model
model_results = []
predictions = {}

for name, model in models.items():
    
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    predictions[name] = test_pred

    model_results.append({
        "Model": name,
        "Train MSE": mean_squared_error(y_train, train_pred),
        "Test MSE": mean_squared_error(y_test, test_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Test R2": r2_score(y_test, test_pred)})

model_results = pd.DataFrame(model_results)
model_results = model_results.merge(cv_results, on="Model")

In [ ]:
#mean baseline from training data
train_mean = y_train.mean(axis=0).to_numpy()

mean_train_pred = np.tile(train_mean, (len(y_train), 1))
mean_test_pred = np.tile(train_mean, (len(y_test), 1))

baseline = pd.DataFrame([{
    "Model": "Mean Baseline",
    "Train MSE": mean_squared_error(y_train, mean_train_pred),
    "Test MSE": mean_squared_error(y_test, mean_test_pred),
    "Test MAE": mean_absolute_error(y_test, mean_test_pred),
    "Test R2": r2_score(y_test, mean_test_pred),
    "CV MSE": np.nan}])

model_results = pd.concat(
    [baseline, model_results],
    ignore_index=True).sort_values("Test MSE")

print(model_results.round(2).to_string(index=False))

In [ ]:
#Random Forest results by location
rf_pred = predictions["Random Forest"]

location_mse = mean_squared_error(
    y_test,
    rf_pred,
    multioutput="raw_values")

location_results = pd.DataFrame({
    "Location": locations,

    "MAE": mean_absolute_error(
        y_test,
        rf_pred,
        multioutput="raw_values"),

    "MSE": location_mse,

    "R2": r2_score(
        y_test,
        rf_pred,
        multioutput="raw_values")})

location_results["Label"] = (
    location_results["Location"]
    .str.replace("_", " ")
    .str.title())

location_results = location_results.sort_values("MAE").reset_index(drop=True)

print(location_results[["Label", "MAE", "MSE", "R2"]].round(2).to_string(index=False))

In [ ]:
#Comparison (Most, Least)
print("Most predictable location:")
print(location_results.head(1)[["Label", "MAE", "MSE", "R2"]].round(2).to_string(index=False))

print("\nLeast predictable location:")
print(location_results.tail(1)[["Label", "MAE", "MSE", "R2"]].round(2).to_string(index=False))

In [ ]:
#Model comparision graph
plt.figure(figsize=(7, 4))
plt.bar(model_results["Model"], model_results["Test MSE"])
plt.ylabel("Test MSE")
plt.title("Model Comparison")
plt.tight_layout()
plt.show()

In [ ]:
#prediction error by location
plt.figure(figsize=(8, 6))
plt.barh(location_results["Label"], location_results["MAE"])
plt.xlabel("Mean Absolute Error")
plt.title("Random Forest Prediction Error by Location")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.dates as mdates

#actual vs predicted
best_location = location_results.iloc[0]["Location"]
worst_location = location_results.iloc[-1]["Location"]

for location in [best_location, worst_location]:
    i = locations.index(location)
    label = location.replace("_", " ").title()

    plt.figure(figsize=(10, 4))
    plt.plot(dates_test, y_test.iloc[:, i], label="Actual")
    plt.plot(dates_test, rf_pred[:, i], label="Predicted")

    plt.xlabel("Month/Year")
    plt.ylabel("Mean Temperature")
    plt.title(f"Actual vs Predicted Temperature: {label}")
    plt.legend()

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()